# Sample Descriptives 

In [1]:
import pandas as pd
import numpy as np
from os.path import join
import os

from dotenv import load_dotenv
load_dotenv()  

# Load environment variables
path = os.environ['DATA_DIRECTORY']
TOKEN_BALANCE_TABLE_INPUT_PATH = join(path, "../data/snapshot_token_balance_tables_enriched")
VALIDATED_PROJECTIONS_INPUT_PATH = join(path, '../data/validated_token_projection_graphs')


covalent_key = os.environ['COVALENTHQ_API_KEY']
df_snapshots = pd.read_csv('../data/snapshot_selection.csv')
df_tokens = pd.read_csv("../data/final_token_selection.csv")


In [5]:
ddf = pd.read_csv(join(TOKEN_BALANCE_TABLE_INPUT_PATH, f'token_holder_snapshot_balance_labelled_14779829.csv'), index_col=0)


In [6]:
ddf.head(5)

,address,token_address,value,address_checksum,code,label,pct_supply
0,0x000000000000000000000000000000000000000f,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,4.232260e+13,0x000000000000000000000000000000000000000F,b'',NaN,2.645163e-12
1,0x0000000000000000000000000000000000000ad3,0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2,3.114652e+16,0x0000000000000000000000000000000000000aD3,b'',NaN,3.052718e-08
2,0x0000000000000000000000000000000000001337,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,4.000109e+20,0x0000000000000000000000000000000000001337,b'',NaN,4.000331e-07
3,0x0000000000000000000000000000000000021b22,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,3.000000e+20,0x0000000000000000000000000000000000021B22,b'',NaN,3.000167e-07
4,0x0000000000000000000000000000000000080085,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,2.560028e+15,0x0000000000000000000000000000000000080085,b'',NaN,2.560170e-12


In [3]:
from collections import Counter

# Token list (ensures consistency across all snapshots)
selected_tokens = set(df_tokens['address'])

wallet_token_counts = {}

for snapshot in df_snapshots[df_snapshots['Block Height'] > 11547458]['Block Height']:
    ddf = pd.read_csv(join(TOKEN_BALANCE_TABLE_INPUT_PATH, f'token_holder_snapshot_balance_labelled_{snapshot}.csv'))
    ddf = ddf[ddf['token_address'].isin(selected_tokens)]
    ddf = ddf[ddf['value'] > 0]
    
    pivot = ddf.assign(value=1).pivot_table(index='address', columns='token_address', values='value', fill_value=0)
    token_counts = pivot.sum(axis=1)
    wallet_token_counts[snapshot] = token_counts.value_counts().sort_index()

df_wallet_token_distribution = pd.DataFrame(wallet_token_counts).fillna(0).astype(int)
df_wallet_token_distribution.index.name = "# Tokens per Wallet"
df_wallet_token_distribution


,11659570,11861210,12043054,12244515,12438842,12638919,12831436,13029639,13230157,13422506,13620205,13809597,14009885,14210564,14391029,14589816,14779829,14967365
# Tokens per Wallet,,,,,,,,,,,,,,,,,,
1,305271,394642,447833,495261,547844,570661,583793,600873,639430,665639,695451,715212,733211,745304,755891,761608,771754,779943
2,40877,55994,62612,64695,70051,73397,76356,77853,82687,85177,87106,88243,89407,90470,91697,92882,93883,95589
3,8304,11971,13884,14622,15910,16775,17580,18472,19707,20357,21365,21747,22029,22147,22379,22665,22900,23416
4,3004,4369,5670,5848,6277,6706,7095,7459,7937,8121,8262,8475,8530,8610,8682,9018,9120,9200
5,1339,1974,2197,2370,2569,2692,2902,3041,3298,3337,3391,3449,3428,3415,3443,3583,3601,3719
6,723,1027,1275,1349,1434,1505,1608,1718,1806,1847,1852,1874,1871,1824,1827,1925,1968,2002
7,339,484,539,597,668,685,746,797,861,903,931,898,905,911,929,976,969,1012
8,65,126,173,204,257,259,294,311,354,428,432,426,444,435,452,513,501,524
9,39,70,114,137,168,189,197,233,242,237,258,266,253,261,272,273,279,273


In [4]:
import networkx as nx

graph_stats = []

for snapshot in df_snapshots[df_snapshots['Block Height'] > 11547458]['Block Height']:
    path = join(VALIDATED_PROJECTIONS_INPUT_PATH, f'validated_token_projection_graph_{snapshot}.graphml')
    if not os.path.exists(path):
        continue
    
    G = nx.read_graphml(path)
    edge_count = G.number_of_edges()
    node_count = G.number_of_nodes()
    density = nx.density(G)
    largest_cc_size = len(max(nx.connected_components(G), key=len)) if edge_count > 0 else 0

    graph_stats.append({
        "Block Height": snapshot,
        "Edges": edge_count,
        "Nodes": node_count,
        "Density": density,
        "Largest CC Size": largest_cc_size
    })

df_graph_stats = pd.DataFrame(graph_stats).set_index("Block Height")
df_graph_stats


,Edges,Nodes,Density,Largest CC Size
Block Height,,,,
11659570,17,7,0.809524,7
11861210,15,7,0.714286,7
12043054,15,7,0.714286,7
12244515,22,11,0.400000,11
12438842,23,11,0.418182,11
12638919,21,12,0.318182,8
12831436,18,10,0.400000,8
13029639,18,10,0.400000,8
13230157,22,12,0.333333,12


In [7]:
dict(ddf.groupby('token_address').address.count())

{'0x0bc529c00c6401aef6d220be8c6ea1667f6ad93e': np.int64(48502),
 '0x111111111117dc0aa78b770fa6a738034120c302': np.int64(88043),
 '0x1a4b46696b2bb4794eb3d4c26f1c55f9170fa4c5': np.int64(17932),
 '0x1f9840a85d5af5bf1d1762f925bdaddc4201f984': np.int64(308751),
 '0x4e3fbd56cd56c3e72c1403e103b45db9da5b9d2b': np.int64(12244),
 '0x5a98fcbea516cf06857215779fd812ca3bef1b32': np.int64(15269),
 '0x6b3595068778dd592e39a122f4f5a5cf09c90fe2': np.int64(96736),
 '0x6f40d4a6237c257fff2db00fa0510deeecd303eb': np.int64(3907),
 '0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9': np.int64(110672),
 '0x92d6c1e31e14520e676a687f0a93788b716beff5': np.int64(21504),
 '0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2': np.int64(83436),
 '0xba100000625a3754423978a60c9317c58a424e3d': np.int64(41068),
 '0xc00e94cb662c3520282e6f5717214004a7f26888': np.int64(191792),
 '0xd533a949740bb3306d119cc777fa900ba034cd52': np.int64(70844)}

In [11]:
token_holder = {}

for _, row in df_snapshots[df_snapshots['Block Height'] > 11547458].iterrows():
        snapshot_date = row['Date']
        snapshot_block_height = row['Block Height']

        # print(f"Snapshot for Block Height: {snapshot_block_height} - {datetime.datetime.now()}")

        # Load and prepare data
        ddf = pd.read_csv(join(TOKEN_BALANCE_TABLE_INPUT_PATH, f'token_holder_snapshot_balance_labelled_{snapshot_block_height}.csv'))
        
        token_holder[f'{snapshot_block_height}']= dict(ddf.groupby('token_address').address.count())

In [14]:
df_token_holders = pd.DataFrame(token_holder)

token_lookup = df_tokens[['address', 'symbol']].set_index('address')['symbol'].to_dict()


In [16]:
df_token_holders.index = df_token_holders.index.map(token_lookup)


In [17]:
df_token_holders

,11659570,11861210,12043054,12244515,12438842,12638919,12831436,13029639,13230157,13422506,13620205,13809597,14009885,14210564,14391029,14589816,14779829,14967365
YFI,22023.0,26634.0,31814.0,36132.0,36143.0,38136.0,39355.0,40526.0,41214.0,41848.0,44198.0,44964.0,46513.0,47164.0,47612.0,48118.0,48503.0,48928
1INCH,18860.0,42421.0,49756.0,52245.0,61377.0,65079.0,67425.0,70464.0,71997.0,74099.0,78145.0,81673.0,83509.0,84542.0,86142.0,87036.0,88045.0,89017
UNI,121990.0,161264.0,191620.0,214695.0,247062.0,251398.0,255956.0,265111.0,271551.0,278887.0,288434.0,294203.0,298063.0,301233.0,304860.0,306336.0,308753.0,310348
LDO,786.0,1737.0,2158.0,2580.0,3586.0,4121.0,4737.0,5585.0,6946.0,10167.0,11097.0,11499.0,11787.0,12049.0,12456.0,13687.0,15269.0,16119
SUSHI,26553.0,35033.0,40971.0,44880.0,50704.0,56315.0,60421.0,64650.0,71979.0,76430.0,81481.0,85060.0,89356.0,91970.0,93824.0,95250.0,96738.0,98231
AAVE,28959.0,56497.0,68798.0,72181.0,79198.0,82700.0,86495.0,91351.0,95422.0,98578.0,102364.0,103762.0,105501.0,107017.0,108345.0,108772.0,110673.0,113706
MKR,60418.0,71860.0,75595.0,78285.0,81094.0,81329.0,81256.0,79049.0,79936.0,80630.0,81540.0,81587.0,81971.0,82015.0,82102.0,82666.0,83438.0,84343
BAL,24886.0,27799.0,31553.0,35178.0,35309.0,36206.0,37146.0,38270.0,38734.0,39118.0,39618.0,39757.0,39937.0,39977.0,40223.0,40912.0,41070.0,41592
COMP,117106.0,135426.0,144625.0,153858.0,164209.0,172245.0,174388.0,173723.0,178051.0,182035.0,185587.0,186308.0,187459.0,188378.0,189487.0,190784.0,191794.0,193630
CRV,16689.0,22519.0,25450.0,28985.0,31957.0,36100.0,38851.0,41618.0,44204.0,48513.0,51784.0,56359.0,61203.0,64576.0,66811.0,68896.0,70846.0,72420
